# Backtest do custo do atraso no VaR

Estima a lei L(d) = c * d^beta do custo de receber o VaR com atraso d (etapa 4 da metodologia em `artigo.md`). Usa as barras de 1 minuto de `../dados/barras_1m.csv.gz`, geradas por `baixar_dados.py`.

**Carteira.** Por ação, 5 straddles vendidos (K = 10 opções) com strikes de 90% a 110% do preço de abertura, vencimento de 3 meses e o mesmo nocional por ação. O delta é zerado a cada minuto com a própria ação. A carteira é montada na abertura e encerrada no fechamento, então cada dia é um episódio independente, sem salto noturno nem vencimento no meio.

**Regra.** A cada minuto t, a mesa recebe o VaR 99% de 1 dia calculado com os preços de t - d. Se ele passa do limite, a carteira inteira é reduzida pelo fator limite / VaR, o que devolve o VaR estimado exatamente ao limite. O limite é o percentil 90 do VaR sem atraso. A posição só diminui ao longo do dia.

**Custo do atraso.** L(d) = perda média diária com atraso d menos a mesma com d = 0, em dólares por US$1 milhão de limite de VaR: o custo esperado do atraso. Como medida de cauda, L_es(d) é o aumento do ES de 97,5% da perda diária, a medida de risco do FRTB de Basileia. Com recálculo a cada minuto, L por recálculo é L por dia dividido pelo número de minutos do pregão.

**VaR do backtest e referência de tempo.** E2 usa o VaR delta-gama com 1.000 cenários como indicador da regra de corte; E1/E3 usam reavaliação completa. Com delta zerado, a aproximação retém o termo quadrático da perda local, mas omite termos de ordem superior e a passagem do tempo. A normalização pelo VaR de abertura remove uma escala multiplicativa constante, não o erro variável da aproximação. Como o corte é não linear, esse erro não se cancela necessariamente em L(d).

A combinação com o tempo de E1 constitui, portanto, um cenário: supõe que o indicador aproximado representa as decisões que seriam tomadas com reavaliação completa. Não se afirma ter medido o custo do atraso deste último método. Essa distinção é mantida na metodologia, na discussão e nas conclusões do artigo. E4 compara também outras formas de L, sem apresentar extrapolações subminuto como observações.

**Escolha do limite.** O percentil 90 do VaR sem atraso é um gatilho relativo, escolhido para que existam episódios de corte sem manter a mesa constantemente limitada. Antes de qualquer redução, aproximadamente 10% das observações da distribuição de referência ficam acima dele (empates e a redução cumulativa podem alterar os cortes efetivos). Essa escolha determina a frequência de intervenção e a escala de L; os resultados são condicionais a ela e não representam um limite regulatório calibrado.


In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import ndtr

ATRASOS = [0, 1, 2, 5, 10, 30]                      # minutos
NS = [1, 2, 4, 8, 16, 32, 64]
SORTEIOS = 20                                       # carteiras sorteadas por N; N=1 usa cada ação e N=64 usa todas
STRIKES = np.array([0.90, 0.95, 1.00, 1.05, 1.10])  # 5 straddles vendidos por ação: K = 10
VENC = 0.25                                         # vencimento das opções, em anos
R = 0.04                                            # taxa livre de risco anual
JANELA_VOL = 20                                     # dias para a volatilidade diária
JANELA_CORR = 60                                    # dias para a correlação
MEIA_VIDA = 30                                      # minutos, volatilidade intradiária por média exponencial
PCT_LIMITE = 90                                     # limite no percentil 90 do VaR sem atraso
NIVEL_ES = 0.975                                    # expected shortfall, como no FRTB
M = 1000                                            # cenários do VaR delta-gama
rng = np.random.default_rng(0)

## Dados

Pregão regular (9:30 às 16:00 de Nova York). Minuto sem negócio fica com o último preço do dia.

In [2]:
df = pd.read_csv("../dados/barras_1m.csv.gz", usecols=["acao", "t", "c"])
df["t"] = pd.to_datetime(df["t"], utc=True).dt.tz_convert("America/New_York")
minuto = df["t"].dt.hour * 60 + df["t"].dt.minute
df = df[(minuto >= 570) & (minuto < 960)]
precos = df.pivot(index="t", columns="acao", values="c")
dia = precos.index.normalize()
precos = precos.groupby(dia).ffill().groupby(dia).bfill()
ACOES = list(precos.columns)

dias = [g.to_numpy() for _, g in precos.groupby(dia)]       # um array (minutos x ações) por dia
fech = np.array([d[-1] for d in dias])
ret_d = np.log(fech[1:] / fech[:-1])                        # ret_d[j-1]: do fechamento de j-1 ao de j
var1m = np.array([np.mean(np.diff(np.log(d), axis=0) ** 2, axis=0) for d in dias])
PRIMEIRO = JANELA_CORR + 1                                  # primeiro dia com janelas completas
print(len(ACOES), "ações,", len(dias), "dias,", len(dias) - PRIMEIRO, "dias no backtest,", int(np.isnan(precos.to_numpy()).sum()), "preços faltando")

64 ações, 500 dias, 439 dias no backtest, 0 preços faltando


## Quanto cada ação varia com o tempo

Desvio do retorno em janelas de d minutos dentro do mesmo dia, e o expoente H de desvio ~ d^H. H = 0,5 é o passeio aleatório.

In [3]:
D_H = [1, 2, 5, 10, 30]
soma, cont = np.zeros((len(D_H), len(ACOES))), np.zeros(len(D_H))
for d_ in dias:
    lp = np.log(d_)
    for k, d in enumerate(D_H):
        r = lp[d:] - lp[:-d]
        soma[k] += (r ** 2).sum(0)
        cont[k] += len(r)
H = pd.Series(np.polyfit(np.log(D_H), np.log(np.sqrt(soma / cont[:, None])), 1)[0], index=ACOES)
print(H.describe().round(3).to_string())

count    64.000
mean      0.464
std       0.010
min       0.446
25%       0.457
50%       0.465
75%       0.473
max       0.483


## Carteira unitária de um dia

In [4]:
def straddle(S, K, tau, sig):
    """Preço, delta e gama de um straddle (call + put) por Black-Scholes."""
    v = sig * np.sqrt(tau)
    d1 = (np.log(S / K) + (R + sig**2 / 2) * tau) / v
    desc = K * np.exp(-R * tau)
    call = S * ndtr(d1) - desc * ndtr(d1 - v)
    return 2 * call - S + desc, 2 * ndtr(d1) - 1, 2 * np.exp(-d1**2 / 2) / (np.sqrt(2 * np.pi) * S * v)


def raiz_psd(C):
    """Raiz de uma matriz de correlação; funciona mesmo sem posto completo."""
    w, V = np.linalg.eigh(C)
    return V * np.sqrt(np.clip(w, 0, None))


def dia_unitario(j, idx):
    """VaR por minuto (1 na abertura) e resultado minuto a minuto, já com hedge de delta, de um straddle por strike."""
    S = dias[j][:, idx]
    T = len(S)
    sig = ret_d[j - JANELA_VOL - 1:j - 1, idx].std(0, ddof=1) * np.sqrt(252)
    C = np.corrcoef(ret_d[j - JANELA_CORR - 1:j - 1, idx].T).reshape(len(idx), len(idx))
    tau = VENC - np.arange(T) / (390 * 252)
    preco, delta, gama = straddle(S[:, :, None], S[0][:, None] * STRIKES, tau[:, None, None], sig[:, None])
    qtd = 1 / S[0]                                              # mesmo nocional por ação
    preco, delta, gama = (x.sum(2) * qtd for x in (preco, delta, gama))
    # straddles vendidos (-preço) mais delta comprado em ações; posição de t vale de t a t+1
    u = (-(preco[1:] - preco[:-1]) + delta[:-1] * (S[1:] - S[:-1])).sum(1)
    # volatilidade intradiária: média exponencial de r^2 de 1 minuto, relativa à média dos dias anteriores
    base = var1m[j - JANELA_VOL:j, idx].mean(0)
    r2 = np.vstack([base, np.diff(np.log(S), axis=0) ** 2])
    mult = np.sqrt(pd.DataFrame(r2).ewm(halflife=MEIA_VIDA, adjust=False).mean().to_numpy() / base)
    # VaR delta-gama: com delta zerado, a perda é 1/2 gama dS^2 em cada ação
    a = 0.5 * gama * S**2 * (sig * mult) ** 2 / 252                     # T x N
    Y = np.random.default_rng(j).standard_normal((M, len(idx))) @ raiz_psd(C).T
    perdas = (Y**2) @ a.T                                               # M x T
    k = int(0.99 * M)
    var = np.partition(perdas, k, axis=0)[k]
    return var / var[0], u / var[0]


def resultado_diario(v, u, limite):
    """Resultado do dia para cada atraso. s_t é o fator da carteira; só cai, quando o VaR defasado passa do limite."""
    t = np.arange(len(u))
    return [np.minimum.accumulate(np.minimum(1.0, limite / v[np.maximum(t - d, 0)])) @ u for d in ATRASOS]

## Backtest

N = 1 roda cada ação sozinha (o backtest por ação). N de 2 a 32 usa 20 sorteios das 64 ações, e N = 64 usa todas.

In [5]:
carteiras = [[i] for i in range(len(ACOES))]
for N in NS[1:-1]:
    carteiras += [sorted(rng.choice(len(ACOES), N, replace=False)) for _ in range(SORTEIOS)]
carteiras.append(list(range(len(ACOES))))

linhas = []
t0 = time.time()
for c, idx in enumerate(carteiras):
    vu = [dia_unitario(j, idx) for j in range(PRIMEIRO, len(dias))]
    limite = np.percentile(np.concatenate([v for v, _ in vu]), PCT_LIMITE)
    res = np.array([resultado_diario(v, u, limite) for v, u in vu]) / limite * 1e6   # dias x atrasos, US$ por US$1 milhão de limite
    perda = -res
    q = np.quantile(perda, NIVEL_ES, axis=0)
    for k, d in enumerate(ATRASOS):
        linhas.append({"carteira": c, "N": len(idx), "acoes": " ".join(ACOES[i] for i in idx), "atraso_min": d,
                       "es": perda[perda[:, k] >= q[k], k].mean(), "media": res[:, k].mean(), "desvio": res[:, k].std()})
    if c + 1 == len(carteiras) or len(carteiras[c + 1]) != len(idx):
        print(f"N={len(idx)} pronto, {time.time() - t0:.0f} s")

res = pd.DataFrame(linhas)
# L: perda média a mais por dia (custo esperado, métrica principal); L_es: aumento do ES 97,5% (risco de cauda)
res["L"] = res.groupby("carteira")["media"].transform("first") - res["media"]
res["L_es"] = res["es"] - res.groupby("carteira")["es"].transform("first")
res.to_csv("../dados/backtest_L.csv", index=False)

N=1 pronto, 40 s


N=2 pronto, 56 s


N=4 pronto, 77 s


N=8 pronto, 103 s


N=16 pronto, 134 s


N=32 pronto, 176 s


N=64 pronto, 179 s


## Ajuste de L(d) = c * d^beta

Para cada carteira, regressão de log L em log d (d em segundos), usando os atrasos com L > 0; o mesmo para L_es. c sai em dólares por dia, por US$1 milhão de limite, com d em segundos.

In [6]:
def ajuste(g):
    out = {}
    for col, suf in (("L", ""), ("L_es", "_es")):
        h = g[(g["atraso_min"] > 0) & (g[col] > 0)]
        if len(h) < 3:
            out["c" + suf], out["beta" + suf] = np.nan, np.nan
            continue
        beta, log_c = np.polyfit(np.log(h["atraso_min"] * 60), np.log(h[col]), 1)
        out["c" + suf], out["beta" + suf] = np.exp(log_c), beta
    return pd.Series(out)

cb = res.groupby(["carteira", "N", "acoes"]).apply(ajuste, include_groups=False).reset_index()
cb.to_csv("../dados/backtest_c_beta.csv", index=False)
resumo = cb.groupby("N").agg(carteiras=("beta", "size"), sem_ajuste=("beta", lambda x: x.isna().sum()),
                             beta_mediana=("beta", "median"), beta_q25=("beta", lambda x: x.quantile(0.25)),
                             beta_q75=("beta", lambda x: x.quantile(0.75)), c_mediana=("c", "median"),
                             beta_es_mediana=("beta_es", "median"), c_es_mediana=("c_es", "median"))
print(resumo.round(3).to_string())

    carteiras  sem_ajuste  beta_mediana  beta_q25  beta_q75  c_mediana  beta_es_mediana  c_es_mediana
N                                                                                                    
1          64           0         0.687     0.605     0.729     20.496            0.699       250.134
2          20           0         0.670     0.621     0.710     21.108            0.685       246.267
4          20           0         0.699     0.642     0.745     19.784            0.767       147.772
8          20           0         0.720     0.689     0.738     16.440            0.775       125.616
16         20           0         0.732     0.718     0.747     13.879            0.818        69.474
32         20           0         0.744     0.736     0.752     13.183            0.797        75.852
64          1           0         0.758     0.758     0.758     11.527            0.837        56.963


In [ ]:
# As figuras do artigo são geradas centralmente por figuras_artigo.py,
# a partir dos CSVs, sem a caixa de uma única carteira ou painéis redundantes.
um = cb[cb["N"] == 1].set_index("acoes")
correlacao_beta_H = H.loc[um.index].corr(um["beta"])
pd.DataFrame({"acao": um.index, "H": H.loc[um.index].to_numpy(),
              "beta": um["beta"].to_numpy()}).to_csv("../dados/expoentes_H_beta.csv", index=False)
print(f"Correlação de Pearson entre H e beta: {correlacao_beta_H:.4f}")


## Interpretação e limites

- L(d) é medido somente nos atrasos de 1 a 30 minutos; subminuto permanece um cenário em E4.
- O expoente beta descreve o resultado da regra de corte e do hedge, não apenas o movimento do preço H.
- O limite é um gatilho relativo no percentil 90. Os números não são universais para outras regras de intervenção.
- O indicador delta-gama não equivale à reavaliação completa, e seus erros não se cancelam necessariamente na diferença de resultados.
